In [40]:
import pandas as pd
import matplotlib.pyplot as plt
import os
import numpy as np
from matplotlib.ticker import PercentFormatter

In [41]:
def find_files_with_extension(path, extension):
    matching_files = []
    for file in os.listdir(path):
        if file.endswith(extension):
            matching_files.append(os.path.join(path, file)) # creates the full path
    return matching_files

In [ ]:
main_path = "/Users/emluu/Documents/Siegel lab/standard/Rosetta Ligand/Laccases/C12 docking/"

filetag = 'measurements.csv' # usually '.csv'
file_column = 'fileName'
target_column = 'AFB1'
num_bins = 16
global_min = 2
global_max = 6
y_max = 12
bin_edges = np.linspace(global_min, global_max, num_bins + 1)

[2.   2.25 2.5  2.75 3.   3.25 3.5  3.75 4.   4.25 4.5  4.75 5.   5.25
 5.5  5.75 6.  ]


In [58]:
items = os.listdir(main_path)
folders = [item for item in items if os.path.isdir(os.path.join(main_path,item))]
results = []
for folder in folders: 
    current_filename = find_files_with_extension(os.path.join(main_path,folder), filetag)
    current_data = pd.read_csv(current_filename[0])
    current_data = current_data.dropna(axis=1, how='all')
    current_columns = current_data.columns.to_list()
    current_columns.remove(file_column)

    specific_columns = []
    for item in current_columns:
        if target_column in item:
            specific_columns.append(item)

    for column in specific_columns:
        # Calculate histogram values with weights
        counts, bins = np.histogram(
            current_data[column],
            bins=bin_edges,
            weights=np.ones(len(current_data)) / len(current_data)
        )
        # Find the bin with the highest peak
        max_index = np.argmax(counts)
        max_bin_value = bins[max_index]
        max_count = counts[max_index]
        # Store the result
        results.append({
            'name': folder,
            'column': column,
            'peak_bin_value': max_bin_value,
            'peak_percentage': max_count
        })

# Convert to DataFrame and save
results_df = pd.DataFrame(results)
results_df.to_csv(os.path.join(main_path,'histogram_peaks.csv'), index=False)
    

# Grouped Histogram

In [44]:
group_by = 'name'
plot_column = 'rmsd'
filenames =  find_files_with_extension(main_path, filetag)
for file in filenames:
    current_df = pd.read_csv(os.path.join(main_path, file))
    groups = current_df.groupby(group_by)
    
    # Create subplot grid
    num_groups = len(groups)
    fig = plt.figure()
    ax = plt.gca()
    
    index = 0
    colors = ["#47817E","#c6f07f","#2c8746"]
    # Plot each group in its own subplot
    for group_name, group_data in groups:
        ax.hist(group_data[plot_column], bins=bin_edges, alpha=0.7, edgecolor="white", label=group_name, color=colors[index])
        index +=1
    ax.set_ylabel('Frequency', fontsize=10)
    ax.set_xlim(global_min,global_max)
    ax.set_ylim(0,y_max)

    # Common labels
    #fig.suptitle(f'Distribution of {plot_column} in {file.split('/')[-1]}', y=1.02, fontsize=14)
    ax.set_xlabel(plot_column, fontsize=10)
    plt.tight_layout()
    plt.legend()
    plt.show()